# Klasifikasi: confusion matrix, precision, recall, dan ambang

Baca jenis kesalahan dan pilih metrik berdasarkan kebutuhan keputusan.

[Baca artikel lengkap](https://article.fikss.site/programming/ds-klasifikasi)

## Cara memakai
1. Masuk ke akun Google jika diminta.
2. Pilih **File → Save a copy in Drive / Simpan salinan di Drive** untuk menyimpan perubahan.
3. Jalankan sel dari atas ke bawah dengan tombol ▶ atau Shift+Enter. CPU sudah cukup.
4. Ubah kode contoh, jalankan ulang, lalu kerjakan latihan.

Data seluruhnya fiktif/sintetis. File sesi bersifat sementara; unduh hasil yang ingin disimpan. Jika sesi direset, jalankan ulang semua sel. Versi paket Colab dapat membuat hasil numerik sedikit berbeda dari artikel.


## 1. Siapkan pustaka
Sel ini memasang paket yang belum tersedia. Jika Colab meminta restart sesi setelah pemasangan, restart lalu jalankan lagi dari atas.


In [ ]:
%pip install -q "numpy>=1.26" "pandas>=2.2" "matplotlib>=3.8" "scikit-learn>=1.4"


## 2. Siapkan data latihan
Sel ini membuat dua CSV contoh dan helper pembersihan data pada folder sesi.


In [ ]:
from pathlib import Path
import sys
# Dataset fiktif dan helper disertakan; tidak perlu mengunduh ZIP.
berkas_latihan = {'datasets/monitoring.csv': 'waktu,perangkat,up,rtt_ms\n2026-10-01T00:00:00Z,BTS-A,1,20\n2026-10-01T00:01:00Z,BTS-A,1,25\n2026-10-01T00:02:00Z,BTS-A,0,\n2026-10-01T00:03:00Z,BTS-A,,\n2026-10-01T00:00:00Z,BTS-B,1,35\n2026-10-01T00:01:00Z,BTS-B,1,40\n2026-10-01T00:02:00Z,BTS-B,1,45\n2026-10-01T00:03:00Z,BTS-B,0,\n', 'datasets/penjualan.csv': 'id,tanggal,kota,produk,jumlah,harga_rupiah\nT001,2026-10-01,Surabaya,Router,2,500000\nT002,2026-10-01,Surabaya,Kabel,5,20000\nT003,2026-10-02,Malang,Router,1,500000\nT003,2026-10-02,Malang,Router,1,500000\nT004,2026-10-02,Malang,Kabel,-1,20000\nT005,2026-10-03,Surabaya,Switch,1,300000\nT006,2026-10-03,Surabaya,Kabel,,20000\nT007,2026-10-03,Malang,Kabel,3,20000\n', 'da04_kualitas.py': "from pathlib import Path\nimport numpy as np\nimport pandas as pd\n\ndef bersihkan_penjualan(raw):\n    wajib = {'id', 'tanggal', 'kota', 'produk', 'jumlah', 'harga_rupiah'}\n    if not wajib <= set(raw.columns):\n        raise ValueError('Kolom wajib belum lengkap.')\n    data = raw.drop_duplicates().copy()\n    duplikat_identik = len(raw) - len(data)\n    data['tanggal'] = pd.to_datetime(data['tanggal'], format='%Y-%m-%d', errors='coerce')\n    for col in ['id', 'kota', 'produk']:\n        data[col] = data[col].astype('string').str.strip()\n    for col in ['jumlah', 'harga_rupiah']:\n        data[col] = pd.to_numeric(data[col], errors='coerce')\n    valid = data['tanggal'].notna() & ~data['id'].duplicated(keep=False)\n    for col in ['id', 'kota', 'produk']:\n        valid &= data[col].notna() & data[col].ne('')\n    valid &= np.isfinite(data['jumlah']) & data['jumlah'].between(1, 10000) & data['jumlah'].mod(1).eq(0)\n    valid &= np.isfinite(data['harga_rupiah']) & data['harga_rupiah'].between(0, 1000000000) & data['harga_rupiah'].mod(1).eq(0)\n    valid = valid.fillna(False)\n    diterima = data.loc[valid].copy()\n    ditolak = data.loc[~valid].copy()\n    for col in ['jumlah', 'harga_rupiah']:\n        diterima[col] = diterima[col].astype('int64')\n    diterima['omzet_rupiah'] = diterima['jumlah'] * diterima['harga_rupiah']\n    return diterima, ditolak, duplikat_identik\n\nif __name__ == '__main__':\n    raw = pd.read_csv(Path(__file__).resolve().parent / 'datasets/penjualan.csv', dtype={'id':'string'})\n    baik, buruk, duplikat = bersihkan_penjualan(raw)\n    print('Mentah:', len(raw))\n    print('Duplikat identik:', duplikat)\n    print('Ditolak:', len(buruk))\n    print('Diterima:', len(baik))\n"}
for nama, isi in berkas_latihan.items():
    tujuan = Path.cwd() / nama
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    tujuan.write_text(isi, encoding="utf-8")
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print("Dataset dan helper siap.")


## 3. Contoh: ds08_klasifikasi.py
Kode berikut sama dengan contoh di artikel. Baris `__file__` menyesuaikan lokasi CSV untuk notebook.


In [ ]:
__file__ = str(Path.cwd() / 'ds08_klasifikasi.py')
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
if __name__ == '__main__':
    aktual=np.array([0,0,0,1,1,1])
    skor=np.array([0.1,0.6,0.2,0.4,0.8,0.9])  # Skor fiktif yang sudah disediakan.
    prediksi=(skor>=0.5).astype(int)
    print('Matriks [TN, FP; FN, TP]:', confusion_matrix(aktual,prediksi,labels=[0,1]).tolist())
    print('Precision/recall/F1:', [round(f(aktual,prediksi,zero_division=0),3) for f in [precision_score,recall_score,f1_score]])


### Acuan keluaran
```text
Matriks [TN, FP; FN, TP]: [[2, 1], [1, 2]]
Precision/recall/F1: [0.667, 0.667, 0.667]
```


## 4. Lihat grafik dan hasil
Grafik yang dibuat contoh ditampilkan di bawah. File CSV/PNG tersedia di panel Files Colab untuk diunduh.


In [ ]:
from IPython.display import display, Image
folder_hasil = Path("hasil-latihan")
if folder_hasil.exists():
    for hasil in sorted(folder_hasil.iterdir()):
        if hasil.is_file():
            print(hasil)
            if hasil.suffix.lower() == ".png":
                display(Image(filename=str(hasil)))


## 5. Coba sendiri
1. Buat contoh kelas tidak seimbang yang menunjukkan accuracy tinggi tetapi recall positif nol.
2. Tuliskan kebutuhan keputusan yang lebih menekankan precision dan yang lebih menekankan recall.

In [ ]:
# Tulis percobaan dan jawabanmu di sini.


## Cek pemahaman
Mengapa model yang selalu menebak kelas mayoritas dapat berbahaya?

Kembali ke artikel untuk tes pemahaman dan pembahasan.

**Perhatikan:** Jangan menganggap seluruh skor model sebagai probabilitas terkalibrasi. Evaluasi ambang serta biaya kesalahan secara terpisah sebelum penerapan.
